# Urban Air Watch — Data Cleaning

**Phase 2, Step 1**

This notebook cleans the merged AQI + weather dataset and engineers new features for analysis and modeling.

**Input**  : `data/processed/aqi_weather_merged.csv`
**Output** : `data/processed/aqi_clean.csv`

**Steps:**
1. Load data
2. Drop stale rows (Pune station issue)
3. Drop unnecessary columns
4. Fix data types
5. Handle missing values
6. Remove AQI outliers
7. Feature engineering (time-based + season)
8. Save clean data
9. Summary report

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

## 1. Load Data

In [2]:
INPUT_PATH  = "../data/processed/aqi_weather_merged.csv"
OUTPUT_PATH = "../data/processed/aqi_clean.csv"

df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
df.head()

Loaded 255 rows, 30 columns


,city,recorded_at,fetched_at,is_stale,aqi,aqi_category,dominant_pollutant,pm25,pm10,no2,o3,co,so2,temp_c,feels_like_c,humidity_pct,dew_point_c,pressure_hpa,wind_speed_ms,wind_direction_deg,wind_cardinal,wind_gust_ms,visibility_m,visibility_km,weather_main,weather_desc,cloud_cover_pct,sunrise_utc,sunset_utc,weather_fetched_at
0,Mumbai,2026-04-11 21:00:00,2026-04-11 21:43:05,False,160,Unhealthy,pm25,160,105,2.3,11.5,2.5,11.9,28.99,32.71,70,22.99,1010,3.09,290,W,NaN,4000,4.0,Haze,haze,0,2026-04-11 00:54:34+00:00,2026-04-11 13:24:29+00:00,2026-04-11 21:43:05
1,Delhi,2026-04-11 21:00:00,2026-04-11 21:43:06,False,116,Unhealthy for Sensitive Groups,pm25,116,114,45.2,1.4,8.8,4.2,28.05,27.13,30,14.05,1008,1.54,320,NW,NaN,4500,4.5,Haze,haze,75,2026-04-11 00:30:07+00:00,2026-04-11 13:13:59+00:00,2026-04-11 21:43:06
2,Pune,2021-01-01 02:00:00,2026-04-11 21:43:07,True,59,Moderate,pm25,59,26,3.2,27.7,3.7,6.1,30.08,28.83,30,16.08,1011,3.26,292,W,5.44,10000,10.0,Clear,clear sky,0,2026-04-11 00:50:52+00:00,2026-04-11 13:20:08+00:00,2026-04-11 21:43:07
3,Chennai,2026-04-11 21:00:00,2026-04-11 21:43:08,False,57,Moderate,pm25,57,27,4.9,6.8,4.5,2.3,29.79,36.79,82,26.19,1011,4.12,140,SE,NaN,6000,6.0,Clouds,few clouds,20,2026-04-11 00:28:41+00:00,2026-04-11 12:50:57+00:00,2026-04-11 21:43:08
4,Kolkata,2026-04-11 21:00:00,2026-04-11 21:43:09,False,103,Unhealthy for Sensitive Groups,pm25,103,62,30.6,3.8,10.5,8.0,26.97,27.84,57,18.37,1008,2.57,150,SE,NaN,3500,3.5,Haze,haze,20,2026-04-10 23:50:05+00:00,2026-04-11 12:24:50+00:00,2026-04-11 21:43:10


## 2. Drop Stale Rows

The Pune WAQI station (`@7567`) returns data timestamped January 2021 — over 5 years old.
The pipeline already flags this with `is_stale = True`. We drop these rows entirely
since they don't represent real-time air quality.

In [3]:
stale_count = df["is_stale"].sum()
df = df[df["is_stale"] == False].copy()

print(f"Dropped {stale_count} stale rows")
print(f"Remaining: {len(df)} rows")
df["city"].value_counts()

Dropped 71 stale rows
Remaining: 184 rows


city
Delhi      51
Chennai    48
Kolkata    47
Mumbai     38
Name: count, dtype: int64

## 3. Drop Unnecessary Columns

Removing columns that are either:
- Pipeline metadata not useful for analysis (`fetched_at`, `weather_fetched_at`)
- Redundant with another column (`feels_like_c` ~ `temp_c`, `wind_direction_deg` ~ `wind_cardinal`)
- Not useful for modeling (`sunrise_utc`, `sunset_utc`, `weather_desc`, `visibility_m`)
- No longer needed after filtering (`is_stale`)

In [4]:
DROP_COLS = [
    "fetched_at", "weather_fetched_at", "feels_like_c",
    "sunrise_utc", "sunset_utc", "visibility_m",
    "weather_desc", "wind_direction_deg", "is_stale",
]

df.drop(columns=DROP_COLS, inplace=True)
print(f"Dropped {len(DROP_COLS)} columns")
print(f"Remaining columns ({len(df.columns)}):")
print(df.columns.tolist())

Dropped 9 columns
Remaining columns (21):
['city', 'recorded_at', 'aqi', 'aqi_category', 'dominant_pollutant', 'pm25', 'pm10', 'no2', 'o3', 'co', 'so2', 'temp_c', 'humidity_pct', 'dew_point_c', 'pressure_hpa', 'wind_speed_ms', 'wind_cardinal', 'wind_gust_ms', 'visibility_km', 'weather_main', 'cloud_cover_pct']


## 4. Fix Data Types

Convert `recorded_at` to a proper datetime object so we can extract time-based
features later (hour, day of week, month, etc.).

In [5]:
df["recorded_at"] = pd.to_datetime(df["recorded_at"])

float_cols = ["no2", "o3", "co", "so2", "temp_c", "dew_point_c",
              "wind_speed_ms", "wind_gust_ms", "visibility_km"]

for col in float_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Data types fixed")
df.dtypes

Data types fixed


city                             str
recorded_at           datetime64[us]
aqi                            int64
aqi_category                     str
dominant_pollutant               str
pm25                           int64
pm10                           int64
no2                          float64
o3                           float64
co                           float64
so2                          float64
temp_c                       float64
humidity_pct                   int64
dew_point_c                  float64
pressure_hpa                   int64
wind_speed_ms                float64
wind_cardinal                    str
wind_gust_ms                 float64
visibility_km                float64
weather_main                     str
cloud_cover_pct                int64
dtype: object

## 5. Handle Missing Values

Check which columns have nulls, then fix them with a sensible default —
not just drop rows blindly.

In [6]:
print("Null counts before fixing:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Null counts before fixing:
wind_gust_ms    35
dtype: int64


`wind_gust_ms` is missing whenever the wind is calm (OpenWeatherMap simply
omits the field instead of returning 0). We fill these with `0.0` since a
missing gust value in calm conditions genuinely means "no gust", not "unknown".

In [7]:
gust_nulls = df["wind_gust_ms"].isnull().sum()
df["wind_gust_ms"] = df["wind_gust_ms"].fillna(0.0)

print(f"Filled {gust_nulls} missing wind_gust_ms values with 0.0")
print(f"\nNull counts after fixing:")
print(df.isnull().sum().sum(), "total nulls remaining")

Filled 35 missing wind_gust_ms values with 0.0

Null counts after fixing:
0 total nulls remaining


## 6. Remove AQI Outliers

AQI values above 500 are considered sensor errors (500 is the official US-EPA
maximum scale value). We cap any such values and also check for impossible
negative readings.

In [8]:
print(f"AQI range before cleaning: {df['aqi'].min()} to {df['aqi'].max()}")

outliers_high = (df["aqi"] > 500).sum()
df["aqi"] = df["aqi"].clip(upper=500)

outliers_low = (df["aqi"] < 0).sum()
if outliers_low:
    df = df[df["aqi"] >= 0]

print(f"Capped {outliers_high} values above 500")
print(f"Removed {outliers_low} rows with negative AQI")
print(f"AQI range after cleaning: {df['aqi'].min()} to {df['aqi'].max()}")

AQI range before cleaning: 27 to 163
Capped 0 values above 500
Removed 0 rows with negative AQI
AQI range after cleaning: 27 to 163


## 7. Feature Engineering

Add time-based and seasonal features that will help machine learning models
in Phase 3 detect patterns in pollution levels.

In [9]:
df["hour"]             = df["recorded_at"].dt.hour
df["day_of_week"]       = df["recorded_at"].dt.dayofweek      # 0=Monday, 6=Sunday
df["day_of_week_name"]  = df["recorded_at"].dt.day_name()
df["month"]             = df["recorded_at"].dt.month

print("Added time features: hour, day_of_week, day_of_week_name, month")
df[["recorded_at", "hour", "day_of_week_name", "month"]].head()

Added time features: hour, day_of_week, day_of_week_name, month


,recorded_at,hour,day_of_week_name,month
0,2026-04-11 21:00:00,21,Saturday,4
1,2026-04-11 21:00:00,21,Saturday,4
3,2026-04-11 21:00:00,21,Saturday,4
4,2026-04-11 21:00:00,21,Saturday,4
5,2026-04-11 22:00:00,22,Saturday,4


**Indian Meteorological Seasons**

Using the official IMD season definitions instead of generic Western seasons,
since they map much better to India's actual weather and pollution patterns:
- Winter: Dec, Jan, Feb
- Summer: Mar, Apr, May
- Monsoon: Jun, Jul, Aug, Sep
- Post-Monsoon: Oct, Nov

In [10]:
def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    if month in [3, 4, 5]:
        return "Summer"
    if month in [6, 7, 8, 9]:
        return "Monsoon"
    return "Post-Monsoon"

df["season"] = df["month"].apply(get_season)

print("Added season feature")
df["season"].value_counts()

Added season feature


season
Monsoon    156
Summer      28
Name: count, dtype: int64

**Daytime and Weekend Flags**

- `is_daytime`: pollution behaves differently at night vs day (traffic, temperature inversion)
- `is_weekend`: traffic-related pollution often drops on weekends

In [11]:
df["is_daytime"] = df["hour"].between(6, 18).astype(int)
df["is_weekend"]  = (df["day_of_week"] >= 5).astype(int)

print("Added is_daytime and is_weekend flags")
df[["hour", "is_daytime", "day_of_week_name", "is_weekend"]].head()

Added is_daytime and is_weekend flags


,hour,is_daytime,day_of_week_name,is_weekend
0,21,0,Saturday,1
1,21,0,Saturday,1
3,21,0,Saturday,1
4,21,0,Saturday,1
5,22,0,Saturday,1


## 8. Final Column Order & Save

In [12]:
COL_ORDER = [
    # Identifiers
    "city", "recorded_at",
    # Time features
    "hour", "day_of_week", "day_of_week_name", "month", "season",
    "is_daytime", "is_weekend",
    # AQI
    "aqi", "aqi_category", "dominant_pollutant",
    # Pollutants
    "pm25", "pm10", "no2", "o3", "co", "so2",
    # Weather
    "temp_c", "dew_point_c", "humidity_pct", "pressure_hpa",
    "wind_speed_ms", "wind_gust_ms", "wind_cardinal",
    "visibility_km", "weather_main", "cloud_cover_pct",
]

COL_ORDER = [c for c in COL_ORDER if c in df.columns]
df = df[COL_ORDER]

os.makedirs("../data/processed", exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved clean data to {OUTPUT_PATH}")
print(f"Final shape: {df.shape[0]} rows x {df.shape[1]} columns")

Saved clean data to ../data/processed/aqi_clean.csv
Final shape: 184 rows x 28 columns


## 9. Summary Report

In [13]:
print("=" * 55)
print("  CLEANING SUMMARY")
print("=" * 55)
print(f"Final shape     : {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Cities          : {sorted(df['city'].unique().tolist())}")
print(f"Date range      : {df['recorded_at'].min()} to {df['recorded_at'].max()}")
print(f"Remaining nulls : {df.isnull().sum().sum()}")
print("=" * 55)

  CLEANING SUMMARY
Final shape     : 184 rows x 28 columns
Cities          : ['Chennai', 'Delhi', 'Kolkata', 'Mumbai']
Date range      : 2026-04-11 21:00:00 to 2026-06-24 12:00:00
Remaining nulls : 0


In [14]:
print("Average AQI by city:")
df.groupby("city")["aqi"].agg(["mean", "min", "max", "count"]).round(1)

Average AQI by city:


,mean,min,max,count
city,,,,
Chennai,75.2,55,81,48
Delhi,98.0,39,158,51
Kolkata,82.4,46,109,47
Mumbai,105.3,27,163,38


In [15]:
print("Season distribution:")
df["season"].value_counts()

Season distribution:


season
Monsoon    156
Summer      28
Name: count, dtype: int64

In [16]:
print("Preview of final cleaned dataset:")
df.head(10)

Preview of final cleaned dataset:


,city,recorded_at,hour,day_of_week,day_of_week_name,month,season,is_daytime,is_weekend,aqi,aqi_category,dominant_pollutant,pm25,pm10,no2,o3,co,so2,temp_c,dew_point_c,humidity_pct,pressure_hpa,wind_speed_ms,wind_gust_ms,wind_cardinal,visibility_km,weather_main,cloud_cover_pct
0,Mumbai,2026-04-11 21:00:00,21,5,Saturday,4,Summer,0,1,160,Unhealthy,pm25,160,105,2.3,11.5,2.5,11.9,28.99,22.99,70,1010,3.09,0.0,W,4.0,Haze,0
1,Delhi,2026-04-11 21:00:00,21,5,Saturday,4,Summer,0,1,116,Unhealthy for Sensitive Groups,pm25,116,114,45.2,1.4,8.8,4.2,28.05,14.05,30,1008,1.54,0.0,NW,4.5,Haze,75
3,Chennai,2026-04-11 21:00:00,21,5,Saturday,4,Summer,0,1,57,Moderate,pm25,57,27,4.9,6.8,4.5,2.3,29.79,26.19,82,1011,4.12,0.0,SE,6.0,Clouds,20
4,Kolkata,2026-04-11 21:00:00,21,5,Saturday,4,Summer,0,1,103,Unhealthy for Sensitive Groups,pm25,103,62,30.6,3.8,10.5,8.0,26.97,18.37,57,1008,2.57,0.0,SE,3.5,Haze,20
5,Mumbai,2026-04-11 22:00:00,22,5,Saturday,4,Summer,0,1,153,Unhealthy,pm25,153,92,2.0,11.5,2.0,11.9,28.99,22.99,70,1011,3.09,0.0,W,4.0,Haze,0
6,Delhi,2026-04-11 22:00:00,22,5,Saturday,4,Summer,0,1,132,Unhealthy for Sensitive Groups,pm25,132,114,49.8,0.8,10.6,4.7,28.05,14.05,30,1008,1.54,0.0,NW,4.5,Haze,75
8,Chennai,2026-04-11 22:00:00,22,5,Saturday,4,Summer,0,1,55,Moderate,pm25,55,26,5.1,6.6,4.0,2.5,29.48,25.88,82,1011,4.12,0.0,SE,6.0,Clouds,20
9,Kolkata,2026-04-11 21:00:00,21,5,Saturday,4,Summer,0,1,103,Unhealthy for Sensitive Groups,pm25,103,62,30.6,3.8,10.5,8.0,26.97,18.37,57,1008,2.57,0.0,SE,3.5,Haze,20
10,Mumbai,2026-05-27 20:00:00,20,2,Wednesday,5,Summer,0,0,82,Moderate,pm25,82,48,2.3,11.6,2.4,11.9,30.99,24.99,70,1010,3.09,0.0,W,4.0,Haze,20
11,Delhi,2026-05-27 20:00:00,20,2,Wednesday,5,Summer,0,0,119,Unhealthy for Sensitive Groups,pm10,63,119,43.7,8.0,2.6,6.9,40.05,23.05,15,999,2.06,0.0,W,5.0,Clear,0
